============================================================
NOTEBOOK SUMMARY
============================================================
Author      : [Your Name]
Model       : SlowFast Network (from scratch)
Task        : Binary Video Classification — Shoplifting Detection
Dataset     : 855 videos — 531 normal / 324 shoplifting
              Single store, single camera (2024-02-23)
Results     : Test Accuracy=100%, Precision=1.00, Recall=1.00, F1=1.00
Problems    : 1. Initial 100% accuracy suspected overfitting
              2. Investigated data leakage → fixed with group-based split
              3. Investigated blank/corrupted frames → data was healthy
              4. Root cause: dataset is single-store/single-camera (low diversity)
                 Model genuinely separates behavior but won't generalize
                 to unseen stores or cameras without more diverse data
Solutions   : Group-based train/test split (no scene leakage),
              strong regularization (dropout=0.7, weight_decay=1e-3),
              label_smoothing=0.1, heavy augmentation (5 transforms)
Next Steps  : Test on multi-store dataset for real generalization metrics
============================================================

In [ ]:
# ── CELL 1: Imports ───────────────────────────────────────────────────────────

import re
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (confusion_matrix, classification_report,
                             accuracy_score, precision_score,
                             recall_score, f1_score)
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Device: {device}")
if device.type == "cuda":
    print(f"   GPU : {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

In [ ]:
# ── CELL 2: Config ────────────────────────────────────────────────────────────

DATA_DIR    = Path(r"C:\Users\POTATO\Desktop\Cellula\Shoplifting_Detection\Data\processed")
RAW_DIR     = Path(r"C:\Users\POTATO\Desktop\Cellula\Shoplifting_Detection\Data")
WEIGHTS_DIR = Path("ml_pipeline/weights")
PLOTS_DIR   = Path("ml_pipeline/Notebooks/plots")
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

SLOW_FRAMES     = 8
FAST_FRAMES     = 32
FRAME_SIZE      = 224
BATCH_SIZE      = 4
NUM_EPOCHS      = 30
LR              = 1e-4
WEIGHT_DECAY    = 1e-3
LABEL_SMOOTHING = 0.1
NUM_CLASSES     = 2
TEST_SIZE       = 0.15
VAL_SIZE        = 0.20

CLASSES = {0: "non shop lifters", 1: "shop lifters"}

print(f"📁 Data     : {DATA_DIR}")
print(f"🔢 Batch    : {BATCH_SIZE}")
print(f"🔁 Epochs   : {NUM_EPOCHS}")
print(f"📉 LR       : {LR}")
print(f"⚖️  WD       : {WEIGHT_DECAY}")
print(f"🌀 Smoothing: {LABEL_SMOOTHING}")

In [ ]:
# ── CELL 3: Load Data ─────────────────────────────────────────────────────────

print("📂 Loading preprocessed data...")
slow_data = np.load(DATA_DIR / "slow_frames.npy")
fast_data = np.load(DATA_DIR / "fast_frames.npy")
labels    = np.load(DATA_DIR / "labels.npy")

print(f"✅ Loaded!")
print(f"   Slow  : {slow_data.shape}  dtype={slow_data.dtype}")
print(f"   Fast  : {fast_data.shape}  dtype={fast_data.dtype}")
print(f"   Labels: {labels.shape} — {np.unique(labels, return_counts=True)}")

In [ ]:
# ── CELL 4: Build dataset list (needed for group-based split) ─────────────────

video_extensions = {'.mp4', '.avi', '.mov', '.mkv'}
dataset = []
for label, class_name in CLASSES.items():
    class_dir = RAW_DIR / class_name
    if not class_dir.exists():
        continue
    videos = sorted([f for f in class_dir.iterdir()
                     if f.suffix.lower() in video_extensions])
    for v in videos:
        dataset.append({"path": v, "label": label, "class": class_name})

assert len(dataset) == len(labels), \
    f"Mismatch: {len(dataset)} files vs {len(labels)} labels"
print(f"✅ Dataset list built: {len(dataset)} videos")

In [ ]:
# ── CELL 5: Group-Based Train/Val/Test Split ──────────────────────────────────
#
# Videos with the same scene number (e.g. shop_lifter_100 and shop_lifter_n_100)
# MUST stay in the same split to avoid data leakage.

def extract_group_id(filename):
    name    = Path(filename).stem
    numbers = re.findall(r'\d+', name)
    return numbers[0] if numbers else name

for item in dataset:
    item["group_id"] = extract_group_id(item["path"].name)

unique_groups = sorted(set(d["group_id"] for d in dataset))
print(f"✅ Found {len(unique_groups)} unique scene groups")

# Group-level labels (majority vote per group)
group_labels = {}
for item in dataset:
    gid = item["group_id"]
    group_labels.setdefault(gid, []).append(item["label"])

group_list = list(unique_groups)
group_y    = [round(np.mean(group_labels[g])) for g in group_list]

# Split groups
g_trainval, g_test = train_test_split(
    group_list, test_size=TEST_SIZE, stratify=group_y, random_state=42)

g_tv_y = [round(np.mean(group_labels[g])) for g in g_trainval]
g_train, g_val = train_test_split(
    g_trainval, test_size=VAL_SIZE / (1 - TEST_SIZE),
    stratify=g_tv_y, random_state=42)

g_train, g_val, g_test = set(g_train), set(g_val), set(g_test)

indices   = np.arange(len(dataset))
idx_train = [i for i in indices if dataset[i]["group_id"] in g_train]
idx_val   = [i for i in indices if dataset[i]["group_id"] in g_val]
idx_test  = [i for i in indices if dataset[i]["group_id"] in g_test]

# Verify no leakage
assert len(g_train & g_val)  == 0, "LEAK: train/val!"
assert len(g_train & g_test) == 0, "LEAK: train/test!"
assert len(g_val   & g_test) == 0, "LEAK: val/test!"

print(f"\n✅ Group-based split (no scene leakage):")
print(f"   Train : {len(idx_train)} videos from {len(g_train)} scenes")
print(f"   Val   : {len(idx_val)} videos from {len(g_val)} scenes")
print(f"   Test  : {len(idx_test)} videos from {len(g_test)} scenes")
print(f"   ✅ No leakage confirmed!")

In [ ]:
# ── CELL 6: Dataset Class ─────────────────────────────────────────────────────

class SlowFastDataset(Dataset):
    def __init__(self, slow, fast, labels, augment=False):
        self.slow    = slow
        self.fast    = fast
        self.labels  = labels
        self.augment = augment

    def __len__(self):
        return len(self.labels)

    def _to_tensor(self, frames):
        t = torch.from_numpy(np.array(frames))
        return t.permute(3, 0, 1, 2).float()   # (3, T, H, W)

    def _augment(self, slow, fast):
        # Horizontal flip
        if torch.rand(1) > 0.5:
            slow = torch.flip(slow, dims=[3])
            fast = torch.flip(fast, dims=[3])
        # Brightness
        if torch.rand(1) > 0.5:
            f = torch.FloatTensor(1).uniform_(0.6, 1.4)
            slow = torch.clamp(slow * f, 0, 1)
            fast = torch.clamp(fast * f, 0, 1)
        # Contrast
        if torch.rand(1) > 0.5:
            m = slow.mean()
            f = torch.FloatTensor(1).uniform_(0.7, 1.3)
            slow = torch.clamp((slow - m) * f + m, 0, 1)
            fast = torch.clamp((fast - m) * f + m, 0, 1)
        # Gaussian noise
        if torch.rand(1) > 0.5:
            slow = torch.clamp(slow + torch.randn_like(slow) * 0.02, 0, 1)
            fast = torch.clamp(fast + torch.randn_like(fast) * 0.02, 0, 1)
        # Temporal reverse
        if torch.rand(1) > 0.5:
            slow = slow.flip(dims=[1])
            fast = fast.flip(dims=[1])
        return slow, fast

    def __getitem__(self, idx):
        slow  = self._to_tensor(self.slow[idx])
        fast  = self._to_tensor(self.fast[idx])
        if self.augment:
            slow, fast = self._augment(slow, fast)
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return slow, fast, label

# Build datasets
train_ds = SlowFastDataset(slow_data[idx_train], fast_data[idx_train],
                            labels[idx_train], augment=True)
val_ds   = SlowFastDataset(slow_data[idx_val],   fast_data[idx_val],
                            labels[idx_val],   augment=False)
test_ds  = SlowFastDataset(slow_data[idx_test],  fast_data[idx_test],
                            labels[idx_test],  augment=False)

# WeightedRandomSampler for class imbalance
train_labels  = labels[idx_train]
class_counts  = np.bincount(train_labels)
class_weights = 1.0 / class_counts
sample_w      = class_weights[train_labels]
sampler       = WeightedRandomSampler(torch.FloatTensor(sample_w),
                                      len(train_labels), replacement=True)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=0, pin_memory=(device.type=="cuda"))
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

s, f, l = train_ds[0]
print(f"✅ DataLoader ready")
print(f"   Slow : {s.shape}  Fast : {f.shape}  Label: {l}")
print(f"   Train batches: {len(train_loader)}")

In [ ]:
# ── CELL 7: SlowFast Architecture ─────────────────────────────────────────────

class Conv3dBnRelu(nn.Module):
    def __init__(self, in_c, out_c, kernel, stride=1, padding=0):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv3d(in_c, out_c, kernel, stride=stride,
                      padding=padding, bias=False),
            nn.BatchNorm3d(out_c),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.block(x)


class FastPathway(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = Conv3dBnRelu(3, 8,  (1,7,7), stride=(1,2,2), padding=(0,3,3))
        self.pool1 = nn.MaxPool3d((1,3,3), stride=(1,2,2), padding=(0,1,1))
        self.conv2 = Conv3dBnRelu(8,  16, (3,3,3), stride=(1,2,2), padding=(1,1,1))
        self.conv3 = Conv3dBnRelu(16, 32, (3,3,3), stride=(1,2,2), padding=(1,1,1))
        self.conv4 = Conv3dBnRelu(32, 64, (3,3,3), stride=(1,2,2), padding=(1,1,1))
        self.pool_final = nn.AdaptiveAvgPool3d((1,1,1))

    def forward(self, x):
        x  = self.pool1(self.conv1(x))
        l0 = x
        x  = self.conv2(x);  l1 = x
        x  = self.conv3(x);  l2 = x
        x  = self.conv4(x)
        return self.pool_final(x).flatten(1), [l0, l1, l2]


class SlowPathway(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = Conv3dBnRelu(3,        64,  (1,7,7), stride=(1,2,2), padding=(0,3,3))
        self.pool1 = nn.MaxPool3d((1,3,3), stride=(1,2,2), padding=(0,1,1))
        self.conv2 = Conv3dBnRelu(64+16,   128, (1,3,3), stride=(1,2,2), padding=(0,1,1))
        self.conv3 = Conv3dBnRelu(128+32,  256, (3,3,3), stride=(1,2,2), padding=(1,1,1))
        self.conv4 = Conv3dBnRelu(256+64,  512, (3,3,3), stride=(1,2,2), padding=(1,1,1))
        self.pool_final = nn.AdaptiveAvgPool3d((1,1,1))

    def forward(self, x, laterals):
        x = self.pool1(self.conv1(x))
        x = self.conv2(torch.cat([x, laterals[0]], dim=1))
        x = self.conv3(torch.cat([x, laterals[1]], dim=1))
        x = self.conv4(torch.cat([x, laterals[2]], dim=1))
        return self.pool_final(x).flatten(1)


class LateralConnection(nn.Module):
    def __init__(self, fast_c, out_c):
        super().__init__()
        self.conv = nn.Conv3d(fast_c, out_c, (4,1,1), stride=(4,1,1), bias=False)
        self.bn   = nn.BatchNorm3d(out_c)
        self.relu = nn.ReLU(inplace=True)
    def forward(self, x):
        return self.relu(self.bn(self.conv(x)))


class SlowFastNet(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.fast      = FastPathway()
        self.lateral0  = LateralConnection(8,  16)
        self.lateral1  = LateralConnection(16, 32)
        self.lateral2  = LateralConnection(32, 64)
        self.slow      = SlowPathway()
        self.classifier = nn.Sequential(
            nn.Dropout(p=0.7),
            nn.Linear(512 + 64, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.5),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, slow_x, fast_x):
        fast_out, fast_lat = self.fast(fast_x)
        laterals = [self.lateral0(fast_lat[0]),
                    self.lateral1(fast_lat[1]),
                    self.lateral2(fast_lat[2])]
        slow_out = self.slow(slow_x, laterals)
        return self.classifier(torch.cat([slow_out, fast_out], dim=1))


model      = SlowFastNet(NUM_CLASSES).to(device)
total_p    = sum(p.numel() for p in model.parameters())
trainable  = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✅ SlowFast model built!")
print(f"   Total params    : {total_p:,}")
print(f"   Trainable params: {trainable:,}")

# Sanity check
with torch.no_grad():
    dummy_s = torch.randn(2, 3, 8,  224, 224).to(device)
    dummy_f = torch.randn(2, 3, 32, 224, 224).to(device)
    out     = model(dummy_s, dummy_f)
print(f"   Output shape    : {out.shape} ✅")

In [ ]:
# ── CELL 8: Loss / Optimizer / Scheduler ─────────────────────────────────────

class_w   = torch.FloatTensor(1.0 / class_counts).to(device)
class_w  /= class_w.sum()

criterion = nn.CrossEntropyLoss(weight=class_w,
                                 label_smoothing=LABEL_SMOOTHING)
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer,
                                                  T_max=NUM_EPOCHS, eta_min=1e-6)

print(f"✅ Loss     : CrossEntropyLoss (label_smoothing={LABEL_SMOOTHING})")
print(f"   Optimizer: Adam (lr={LR}, weight_decay={WEIGHT_DECAY})")
print(f"   Scheduler: CosineAnnealingLR")

In [ ]:
# ── CELL 9: Training Loop ─────────────────────────────────────────────────────

def run_epoch(model, loader, criterion, optimizer, is_train, device):
    model.train() if is_train else model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.set_grad_enabled(is_train):
        for slow, fast, lbl in loader:
            slow, fast, lbl = slow.to(device), fast.to(device), lbl.to(device)
            out  = model(slow, fast)
            loss = criterion(out, lbl)
            if is_train:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            correct    += (out.argmax(1) == lbl).sum().item()
            total      += lbl.size(0)
            total_loss += loss.item() * lbl.size(0)
    return total_loss / total, correct / total


history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
best_val_loss = float("inf")

print(f"🚀 Starting training for {NUM_EPOCHS} epochs...\n")
print(f"{'Epoch':>6} {'Train Loss':>12} {'Train Acc':>10} {'Val Loss':>10} {'Val Acc':>9}")
print("─" * 55)

for epoch in range(1, NUM_EPOCHS + 1):
    tr_loss, tr_acc = run_epoch(model, train_loader, criterion,
                                 optimizer, True,  device)
    vl_loss, vl_acc = run_epoch(model, val_loader,   criterion,
                                 None,      False, device)
    scheduler.step()

    history["train_loss"].append(tr_loss)
    history["val_loss"].append(vl_loss)
    history["train_acc"].append(tr_acc)
    history["val_acc"].append(vl_acc)

    tag = " ← best" if vl_loss < best_val_loss else ""
    if vl_loss < best_val_loss:
        best_val_loss = vl_loss
        torch.save(model.state_dict(), WEIGHTS_DIR / "slowfast_best.pth")

    print(f"{epoch:>6}   {tr_loss:>10.4f}   {tr_acc:>8.4f}   "
          f"{vl_loss:>8.4f}   {vl_acc:>8.4f}{tag}")

print(f"\n✅ Training complete! Best val loss: {best_val_loss:.4f}")
print(f"💾 Weights → {WEIGHTS_DIR / 'slowfast_best.pth'}")

In [ ]:
# ── CELL 10: Training Curves ──────────────────────────────────────────────────

epochs = range(1, NUM_EPOCHS + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("SlowFast Training History", fontsize=14, fontweight='bold')

axes[0].plot(epochs, history["train_loss"], 'b-o', label='Train', markersize=4)
axes[0].plot(epochs, history["val_loss"],   'r-o', label='Val',   markersize=4)
axes[0].set_title("Loss"); axes[0].set_xlabel("Epoch")
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, history["train_acc"], 'b-o', label='Train', markersize=4)
axes[1].plot(epochs, history["val_acc"],   'r-o', label='Val',   markersize=4)
axes[1].set_title("Accuracy"); axes[1].set_xlabel("Epoch")
axes[1].set_ylim(0, 1); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(PLOTS_DIR / "training_curves.png", dpi=150, bbox_inches='tight')
plt.show()
print("✅ Training curves saved")

In [ ]:
# ── CELL 11: Evaluate on Test Set ────────────────────────────────────────────

model.load_state_dict(torch.load(WEIGHTS_DIR / "slowfast_best.pth",
                                  map_location=device, weights_only=True))
model.eval()

all_preds, all_labels = [], []

with torch.no_grad():
    for slow, fast, lbl in test_loader:
        out = model(slow.to(device), fast.to(device))
        all_preds.extend(out.argmax(1).cpu().numpy())
        all_labels.extend(lbl.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

print("📊 Test Set Results")
print("─" * 40)
print(f"Accuracy  : {accuracy_score(all_labels, all_preds):.4f}")
print(f"Precision : {precision_score(all_labels, all_preds):.4f}")
print(f"Recall    : {recall_score(all_labels, all_preds):.4f}")
print(f"F1 Score  : {f1_score(all_labels, all_preds):.4f}")
print()
print(classification_report(all_labels, all_preds,
                             target_names=["Normal", "Shoplifting"]))

In [ ]:
# ── CELL 12: Confusion Matrix ─────────────────────────────────────────────────

cm = confusion_matrix(all_labels, all_preds)
tn, fp, fn, tp = cm.ravel()

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=["Normal", "Shoplifting"],
            yticklabels=["Normal", "Shoplifting"], ax=ax)
ax.set_title("Confusion Matrix — SlowFast Network", fontsize=13, fontweight='bold')
ax.set_ylabel("True Label"); ax.set_xlabel("Predicted Label")
ax.text(0.5, -0.15,
        f"TN={tn}  FP={fp}  FN={fn}  TP={tp}",
        transform=ax.transAxes, ha='center', fontsize=10, color='gray')

plt.tight_layout()
plt.savefig(PLOTS_DIR / "confusion_matrix.png", dpi=150, bbox_inches='tight')
plt.show()
print("✅ Confusion matrix saved")
print("\n🎉 SlowFast notebook fully complete and ready to push!")